# Arize Pipeline with DA-Agent Benchmark 

In [1]:
import phoenix as px
import pandas as pd
from phoenix.experiments import run_experiment
import os
import re
import sys
import json
import requests
import zipfile
import io

import nest_asyncio
nest_asyncio.apply()

# Get data for DA-Agent Benchmark

![](https://infiagent.github.io/static/images/framework.png)

You can find more info on these benchmarks at the following links:

https://infiagent.github.io/

https://github.com/InfiAgent/InfiAgent/tree/main

In [2]:
def download_data():
    ''' downloads repo with benchmark data if it does not already exist; returns full path to the folder with data'''
    repo_zip_url = "https://github.com/InfiAgent/InfiAgent/archive/refs/heads/main.zip"
    extract_folder = "da_agent_repo"
    data_relative_path = os.path.join(
        extract_folder, "InfiAgent-main", "examples", "DA-Agent", "data"
    )

    # Check if data folder already exists
    if os.path.exists(data_relative_path):
        print("Data already exists.")
        return os.path.abspath(data_relative_path)

    # If repository folder does not exist, download and extract
    if not os.path.exists(extract_folder):
        print("Downloading and extracting repository...")
        response = requests.get(repo_zip_url)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as zip_file:
            os.makedirs(extract_folder, exist_ok=True)
            zip_file.extractall(path=extract_folder)
        print(f"Repository downloaded and extracted to: {extract_folder}")
    else:
        print("Repository folder exists but data not found; checking contents...")

    # Verify that the data folder was extracted
    if not os.path.exists(data_relative_path):
        raise FileNotFoundError(
            f"Data directory not found at {data_relative_path} after extraction."
        )

    return os.path.abspath(data_relative_path)

path_to_data=download_data()
path_to_data

Data already exists.


'/Users/jrduncan/Desktop/exAI_agent/repo/TACC_exAI/benchmarks/da_agent_repo/InfiAgent-main/examples/DA-Agent/data'

# functions for reading in DA-Agent benchmark 

In [3]:
def read_dicts_from_file(file_name):
    """
    Read a file with each line containing a JSON string representing a dictionary,
    and return a list of dictionaries.

    :param file_name: Name of the file to read from.
    :return: List of dictionaries.
    """
    dict_list = []
    with open(file_name, 'r') as file:
        for line in file:
            # Convert the JSON string back to a dictionary.
            dictionary = json.loads(line.rstrip('\n'))
            dict_list.append(dictionary)
    return dict_list

table_path = os.path.join(path_to_data, "da-dev-tables")
questions = read_dicts_from_file(os.path.join(path_to_data,'da-dev-questions.jsonl'))
solutions=read_dicts_from_file(os.path.join(path_to_data,'da-dev-labels.jsonl'))

In [4]:
df_q = pd.DataFrame(questions)
df_s = pd.DataFrame(solutions)
df_all=pd.merge(df_q, df_s, on='id')
df_all['prompt'] = df_all.apply(lambda row: f"Question: {row['question']} Constraints: {row['constraints']} Data: You can find data relevant to this question in directory /data/{row['file_name']}", axis=1)

In [5]:
df_easy= df_all.iloc[0:10]  #df_all[df_all.level=='easy'].iloc[0:10]
df_easy

,id,question,concepts,constraints,format,file_name,level,common_answers,prompt
0,0,Calculate the mean fare paid by the passengers.,[Summary Statistics],Calculate the mean fare using Python's built-i...,"@mean_fare[mean_fare_value] where ""mean_fare_v...",test_ave.csv,easy,"[[mean_fare, 34.65]]",Question: Calculate the mean fare paid by the ...
1,5,"Generate a new feature called ""FamilySize"" by ...","[Feature Engineering, Correlation Analysis]",Create a new column 'FamilySize' that is the s...,"@correlation_coefficient[r_value]\nwhere ""r_va...",test_ave.csv,medium,"[[correlation_coefficient, 0.21]]","Question: Generate a new feature called ""Famil..."
2,6,"Create a new column called ""AgeGroup"" that cat...","[Feature Engineering, Summary Statistics]",Make sure to round the mean fare of each group...,"@mean_fare_child[mean_fare], @mean_fare_teenag...",test_ave.csv,medium,"[[mean_fare_elderly, 43.47], [mean_fare_teenag...","Question: Create a new column called ""AgeGroup..."
3,7,Apply the linear regression algorithm from the...,[Machine Learning],Use one-hot encoding for the 'Sex' and 'Embark...,"@prediction_accuracy[accuracy], where ""accurac...",test_ave.csv,hard,"[[prediction_accuracy, 0.78]]",Question: Apply the linear regression algorith...
4,8,Perform a distribution analysis on the 'Fare' ...,"[Distribution Analysis, Summary Statistics]",Keep all numerical values rounded to 2 decimal...,"@mean_fare_class1[mean_fare], @median_fare_cla...",test_ave.csv,medium,"[[median_fare_class1, 69.30], [median_fare_cla...",Question: Perform a distribution analysis on t...
5,9,"Calculate the mean value of the ""Close Price"" ...",[Summary Statistics],Use the built-in Python (numpy or pandas) to c...,"@mean_close_price[mean_value], where ""mean_val...",GODREJIND.csv,easy,"[[mean_close_price, 570.68]]","Question: Calculate the mean value of the ""Clo..."
6,10,"Check if the ""Total Traded Quantity"" column ad...",[Distribution Analysis],Use Shapiro-Wilk test from scipy.stats module ...,"@is_normal[response], where ""response"" is a st...",GODREJIND.csv,easy,"[[is_normal, no]]","Question: Check if the ""Total Traded Quantity""..."
7,11,Calculate the correlation coefficient between ...,[Correlation Analysis],Calculate the Pearson correlation coefficient ...,@correlation_coefficient[r_value] @p_value[p_v...,GODREJIND.csv,medium,"[[relationship_type, linear], [correlation_coe...",Question: Calculate the correlation coefficien...
8,14,"Create a new feature called ""Price Range"" whic...","[Feature Engineering, Summary Statistics]",Make sure to use the correct columns for calcu...,@price_range_mean[mean]: The mean should be a ...,GODREJIND.csv,medium,"[[price_range_mean, 16.65], [price_range_std_d...","Question: Create a new feature called ""Price R..."
9,18,Calculate the mean and standard deviation of t...,[Summary Statistics],Outliers are to be pruned via the interquartil...,"@mean_mar_2019[mean] @sd_mar_2019[sd], where ""...",unemployement_industry.csv,easy,"[[mean_mar_2019, 171.44], [sd_mar_2019, 188.25]]",Question: Calculate the mean and standard devi...


In [6]:
df_easy.iloc[4].common_answers

[['median_fare_class1', '69.30'],
 ['median_fare_class2', '15.05'],
 ['std_dev_fare_class1', '80.86'],
 ['mean_fare_class3', '13.23'],
 ['std_dev_fare_class2', '13.19'],
 ['mean_fare_class2', '21.47'],
 ['std_dev_fare_class3', '10.04'],
 ['mean_fare_class1', '87.96']]

In [7]:
{"mean_fare_class1":"87.96","median_fare_class1":"69.30","std_dev_fare_class1":"80.64","mean_fare_class2":"21.47","median_fare_class2":"15.05","std_dev_fare_class2":"13.15","mean_fare_class3":"13.23","median_fare_class3":"8.05","std_dev_fare_class3":"10.03"}

{'mean_fare_class1': '87.96',
 'median_fare_class1': '69.30',
 'std_dev_fare_class1': '80.64',
 'mean_fare_class2': '21.47',
 'median_fare_class2': '15.05',
 'std_dev_fare_class2': '13.15',
 'mean_fare_class3': '13.23',
 'median_fare_class3': '8.05',
 'std_dev_fare_class3': '10.03'}

In [8]:
df_easy.prompt[0]

"Question: Calculate the mean fare paid by the passengers. Constraints: Calculate the mean fare using Python's built-in statistics module or appropriate statistical method in pandas. Rounding off the answer to two decimal places. Data: You can find data relevant to this question in directory /data/test_ave.csv"

In [9]:
df_easy.prompt[1]

'Question: Generate a new feature called "FamilySize" by summing the "SibSp" and "Parch" columns. Then, calculate the Pearson correlation coefficient (r) between the "FamilySize" and "Fare" columns. Constraints: Create a new column \'FamilySize\' that is the sum of \'SibSp\' and \'Parch\' for each row.\nCalculate the Pearson correlation coefficient between \'FamilySize\' and \'Fare\'\nDo not perform any further data cleaning or preprocessing steps before calculating the correlation. Data: You can find data relevant to this question in directory /data/test_ave.csv'

### upload dataset to phoenix

#### start phoenix session for the notebook

Below shows how to configure Arize Phoenix locally with SQLite for persistent data storage on disk, along with setting the relevant environment variables. Can also persist data in postgres. 

```python
# Set environment variable for working directory where SQLite DB will be stored
os.environ['PHOENIX_WORKING_DIR'] = '/path/to/your/phoenix_data'

# Launch Phoenix server with use_temp_dir=False for persistence
px.launch_app(
    use_temp_dir=False,  # Save DB file in PHOENIX_WORKING_DIR instead of temp folder
    working_dir=os.environ['PHOENIX_WORKING_DIR']
)
```

In [10]:
import phoenix as px
from phoenix.experiments import run_experiment

In [11]:
# start phoenix session for the notebook
os.makedirs('phoenix_data', exist_ok=True)
os.environ['PHOENIX_WORKING_DIR'] = 'phoenix_data'
session = px.launch_app(use_temp_dir=False)  # Save DB file in PHOENIX_WORKING_DIR instead of temp folder

# Use code below if you want temporary database 
#session = px.launch_app()

meta_columns = ['id',
 'question',
 'concepts',
 'constraints',
 'format',
 'file_name',
 'level']

px_client = px.Client()

🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
💽 Your data is being persisted to sqlite:///phoenix_data/phoenix.db
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


/Users/jrduncan/Desktop/exAI_agent/venv_exAI/lib/python3.11/site-packages/pydantic/json_schema.py:2324: PydanticJsonSchemaWarning: Default value <phoenix.db.types.db_models.Undefined object at 0x13f51bb10> is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


In [12]:
try:
    dataset = px_client.get_dataset(name="first-da-agent-10")
    print(dataset)
    download_data = False
except Exception as e:
    # Assuming any exception here means the dataset is not found
    download_data = True

if download_data:
    dataset = px_client.upload_dataset(
        dataframe=df_easy,              # pandas DataFrame containing your dataset
        input_keys=["prompt"],   # List of columns to be used as input features (e.g., questions)
        output_keys=["common_answers"],  # List of columns for the expected output/labels (e.g., expected answers)
        metadata_keys=meta_columns,
        dataset_name="first-da-agent-10"  # Name to assign to this dataset in the system
    ) 

Dataset(id='RGF0YXNldDoy', version_id='RGF0YXNldFZlcnNpb246Mg==')


/var/folders/qj/d2mnyrts1kn7sk7g8_jtz7ww0000gp/T/ipykernel_39993/3623125452.py:2: DeprecationWarning: Migrate to using client.datasets.get_dataset via arize-phoenix-client
  dataset = px_client.get_dataset(name="first-da-agent-10")


### Initialize tracer 

In [13]:
trace = True

# YOU CAN OPTIONALLY RUN THIS CELL -- THIS WILL TRACE IN LLM CALLS IN THE EXPERIEMNTS
if trace:
    from openinference.instrumentation.openai import OpenAIInstrumentor
    
    from phoenix.otel import register

    PROJECT_NAME='DA BENCHMARK'
    
    tracer_provider = register(project_name=PROJECT_NAME)
    OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)

    tracer = tracer_provider.get_tracer(__name__)
else:
    tracer=None

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: DA BENCHMARK
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: localhost:4317
|  Transport: gRPC
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



### Define task 

In [14]:
# Assuming the script is in a directory one level up or in a sibling folder
# We should update this to have full path / install script
module_path = os.path.abspath(os.path.join('../'))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from agent import Agent

def task(input,metadata):

    with tracer.start_as_current_span("AgentRun", openinference_span_kind="agent") as span:
        span.set_input(input['prompt'])
        agent = Agent(
       #     mode="dev",
            session=None,
            experiment=True,
            experiment_prompt=input['prompt'],
            data_dir="/Users/jrduncan/Desktop/exAI_agent/repo/InfiAgent/examples/DA-Agent/data/da-dev-tables/",
            tracer=tracer
        )
        
        result=agent.run()
        span.set_attribute(result,"result from agent.run")

        # parse output into json
        output_post_string_parsing,final_result=reformat_output(input,result,metadata) 
        span.set_attribute(output_post_string_parsing,"result from post string parsing")
        span.set_attribute(json.dumps(final_result),"result from reformat_output")
    return final_result

# functions for evaluation 

In [22]:
import re
# stuff I wrote for parsing result returned from agent.run()of code included
def extract_generated_code(text):
    """
    Extracts the Python code block from the provided text by locating start and end markers.

    Returns:
        str or None: The extracted code as a string, or None if not found.
    """
    start_marker = "```"
    end_marker = "```"
    start_index = text.find(start_marker)
    if start_index == -1:
        return None
    start_index += len(start_marker)
    end_index = text.find(end_marker, start_index)
    if end_index == -1:
        return None
    code_block = text[start_index:end_index]
    return code_block.strip()

def check_if_generated_code(output_str):
    if 'Here is the generated code' in output_str:
        return True
    else:
        return False 
'''
def extract_execution_result(text):
    """Extracts the execution result from the program output."""
#    match = re.search(r"Execution result:\n(.*)", text)
    match = re.search(r"Execution result:(.*)", text)
    return match.group(1).strip() if match else None

def extract_execution_result(text):
    match = re.search(r'Execution result:\s*(.*)', text, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ''
'''

def extract_execution_result(text):
    # Match "Execution result:" regardless of case and allow newlines or spaces after it
    pattern = r'Execution\s+result:\s*(.*)'
    match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else ''


# reformat responses; pulled from:
# https://github.com/InfiAgent/InfiAgent/blob/main/examples/DA-Agent/reformat.py

###### Below are prompt templates for formatting final answer so that they can be parsed ###########
####################################################################################################
demons = """\Format{{
@shapiro_wilk_statistic[test_statistic]
@shapiro_wilk_p_value[p_value]
where "test_statistic" is a number between 0 and 1 representing the Shapiro-Wilk test statistic. Rounding off the answer to two decimal places.
where "p_value" is a number between 0 and 1 representing the p-value from the Shapiro-Wilk test. Rounding off the answer to four decimal places.
}}
\Answer{{
@shapiro_wilk_statistic[0.56]
@shapiro_wilk_p_value[0.0002]   
}}

\Format{{
@total_votes_outliers_num[outlier_num]
where "outlier_num" is an integer representing the number of values considered outliers in the 'total_votes' column.
}}
\Answer{{
@total_votes_outliers[10]   
}}
"""

reformat_template = """You should strictly follow the output requirements in the Format part. Here're some examples: 
{demons}. 
Your answer should contain all the \"@answer_name[answer]\" in the order mentioned, each \"answer\" should be in the range of value as required. 
The format requirements of this question is:
{format}. Please give your answer:"""

# functions pulled from eval repo: 
# https://github.com/InfiAgent/InfiAgent/blob/main/examples/DA-Agent/eval_closed_form.py
def extract_format(input_string):
    pattern = r"@(\w+)\[(.*?)\]"
    matches = re.findall(pattern, input_string)
    answer_names = [match[0] for match in matches]
    answers = [match[1] for match in matches]
    return answer_names, answers

def is_equal(response, label):
    if response == label:
        return True
    else:
        try:
            return abs(float(response) - float(label)) < 1e-6 # should this really be this low? 
        except:
            return False

import openai 

api_key_cloud = os.environ.get("EXAI_API_KEY")
api_cloud_endpoint= os.environ.get("EXAI_BASE_URL") 

model = "Qwen3-32B"#"gpt-oss-120b" #"Meta-Llama-3.3-70B-Instruct"
client = openai.OpenAI(api_key=api_key_cloud, base_url = api_cloud_endpoint)

def call(messages):
    return client.chat.completions.create(model=model,messages=messages).choices

def remove_think_block(llm_response: str) -> str:
    """
    Removes any <think> ... </think> content from an LLM response string.
    """
    # Use non-greedy matching to remove only the contents within <think>...</think>
    cleaned_response = re.sub(r"<think>.*?</think>", "", llm_response, flags=re.DOTALL)
    # Optionally strip surrounding whitespace
    return cleaned_response.strip()

@tracer.chain
def reformat_output(input,output,metadata):
    ## extract answer if includes code
    if check_if_generated_code(output): 
        output = extract_execution_result(output) 
        
    ## LLM call to reformat response 
    messages = [{"role": "user", "content":metadata['question']}]   # input['prompt']}]     #metadata['question']}]
    messages.append({"role": "assistant", "content": output})
    messages.append({"role": "user", "content": reformat_template.format(demons=demons, format=metadata['format'])})
    reformatted_response = call(messages)[0].message.content

    ## remove reasoning models thoughts 
    reformatted_response = remove_think_block(reformatted_response)
    
    ## extract correct answwers and put into dictonary 
    answer_names, answers = extract_format(reformatted_response)
    extracted_answers = dict(zip(answer_names, answers))

    return output, extracted_answers

def evaluate_accuracy(output, expected):
    # compare all results to see if they are the same 
    correct_answers = {}
    for answer in expected['common_answers']:
        correct_answers[answer[0]]=False
    for ans_name in output.keys():
        for answer in expected['common_answers']:
            if answer[0] == ans_name:
                correct_answers[ans_name] = is_equal(output[ans_name],answer[1])
    # considered correct if all values are True -- so it gets everything right. 
    if len(correct_answers) == len(expected['common_answers']):
        return all(correct_answers.values())
    else:
        return False 

# Run experiment

In [23]:
import datetime

# Get current date and time
now = datetime.datetime.now()
# Format the date and time
timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")

# Use samba nova
import structured_generation
structured_generation.set_force_local(False)

experiment = run_experiment(
    dataset,
    task=task,
    evaluators=[evaluate_accuracy],
    experiment_name=f"First experiment samba {timestamp}",
    experiment_description="Try changing reformatting model to Qwen3-32B",
    experiment_metadata={'model':"Samba Nova Default", 
                         "prompt_template": None,} ,
#    repetitions=2
#    concurrency=1
#    dry_run=True,
    timeout=1000
)

print("Experiment complete. Check Phoenix UI in your browser.")

🧪 Experiment started.
📺 View dataset experiments: http://localhost:6006/datasets/RGF0YXNldDoy/experiments
🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDo1


running tasks |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s

╭────────────────────────────────────────────── Vector Store Status ──────────────────────────────────────────────╮
│ Building vector store from all session histories.                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── Vector Store Status ──────────────────────────────────────────────╮
│ Vector store build complete.                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Assistant ─────────────────────────────────────────╮                    
│ 👋 How can I assist you today?                                                              │                    
╰─────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────── Plan Execution Progress ───────────────────────────────────────╮     
     │ Plan Progress: 2/2                                                                                    │     
     │                                                                                                       │     
     │ Current Action: SummarizeAndReplyAction                                                               │     
     │                                                                                                       │     
     │ Step Instructions: The mean and standard deviation of the 'Mar.2019' column, after outlier pruning    │     
     │ and handling missing values, are [mean_value] and [std_value], respectively.                          │     
     ╰───────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────── Assistant ─────────────────────────────────────────╮                    
│ The mean of the 'Mar.2019' column, after applying listwise deletion and IQR-based outlier   │                    
│ pruning, is 171.44. The standard deviation is 188.25. Values are rounded to two decimal     │                    
│ places as requested.                                                                        │                    
╰─────────────────────────────────────────────────────────────────────────────────────────────╯

✅ Task runs completed.
🧠 Evaluation started.


running experiment evaluations |          | 0/10 (0.0%) | ⏳ 00:00<? | ?it/s


🔗 View this experiment: http://localhost:6006/datasets/RGF0YXNldDoy/compare?experimentId=RXhwZXJpbWVudDo1

Experiment Summary (10/27/25 03:26 PM -0500)
--------------------------------------------
           evaluator   n  n_scores  avg_score  n_labels  \
0  evaluate_accuracy  10        10        0.8        10   

              top_2_labels  
0  {'True': 8, 'False': 2}  

Tasks Summary (10/27/25 03:25 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0          10      10         0
Experiment complete. Check Phoenix UI in your browser.


In [17]:
# View results in a dataframe 
experiment.as_dataframe()

,output,input,expected,metadata,example_id
run_id,,,,,
RXhwZXJpbWVudFJ1bjo2OA==,"{'mean_mar_2019': '171.44', 'sd_mar_2019': '18...",{'prompt': 'Question: Calculate the mean and s...,"{'common_answers': [['mean_mar_2019', '171.44'...",{'question': 'Calculate the mean and standard ...,RGF0YXNldEV4YW1wbGU6NjA=
RXhwZXJpbWVudFJ1bjo2Nw==,"{'price_range_mean': '16.65', 'price_range_med...",{'prompt': 'Question: Create a new feature cal...,"{'common_answers': [['price_range_mean', '16.6...","{'question': 'Create a new feature called ""Pri...",RGF0YXNldEV4YW1wbGU6NTk=
RXhwZXJpbWVudFJ1bjo2Ng==,"{'correlation_coefficient': '0.99', 'p_value':...",{'prompt': 'Question: Calculate the correlatio...,"{'common_answers': [['relationship_type', 'lin...",{'question': 'Calculate the correlation coeffi...,RGF0YXNldEV4YW1wbGU6NTg=
RXhwZXJpbWVudFJ1bjo2NQ==,{'is_normal': 'no'},"{'prompt': 'Question: Check if the ""Total Trad...","{'common_answers': [['is_normal', 'no']]}","{'question': 'Check if the ""Total Traded Quant...",RGF0YXNldEV4YW1wbGU6NTc=
RXhwZXJpbWVudFJ1bjo2NA==,{'mean_close_price': '570.68'},{'prompt': 'Question: Calculate the mean value...,"{'common_answers': [['mean_close_price', '570....",{'question': 'Calculate the mean value of the ...,RGF0YXNldEV4YW1wbGU6NTY=
RXhwZXJpbWVudFJ1bjo2Mw==,"{'mean_fare_class1': '84.15', 'median_fare_cla...",{'prompt': 'Question: Perform a distribution a...,"{'common_answers': [['median_fare_class1', '69...",{'question': 'Perform a distribution analysis ...,RGF0YXNldEV4YW1wbGU6NTU=
RXhwZXJpbWVudFJ1bjo2Mg==,{'prediction_accuracy': '0.78'},{'prompt': 'Question: Apply the linear regress...,"{'common_answers': [['prediction_accuracy', '0...",{'question': 'Apply the linear regression algo...,RGF0YXNldEV4YW1wbGU6NTQ=
RXhwZXJpbWVudFJ1bjo2MQ==,"{'mean_fare_child': '30.59', 'mean_fare_teenag...",{'prompt': 'Question: Create a new column call...,"{'common_answers': [['mean_fare_elderly', '43....","{'question': 'Create a new column called ""AgeG...",RGF0YXNldEV4YW1wbGU6NTM=
RXhwZXJpbWVudFJ1bjo2MA==,{'correlation_coefficient': '0.21'},{'prompt': 'Question: Generate a new feature c...,{'common_answers': [['correlation_coefficient'...,"{'question': 'Generate a new feature called ""F...",RGF0YXNldEV4YW1wbGU6NTI=


In [20]:
test_ollama = False

In [21]:
# re-run experiment so we can see how two experiments can be run in the UI. 
# Get current date and time
if test_ollama:
    now = datetime.datetime.now()
    # Format the date and time
    timestamp = now.strftime("%Y-%m-%d_%H-%M-%S")
    
    structured_generation.set_force_local(True)
    
    experiment2 = run_experiment(
        dataset,
        task=task,
        evaluators=[evaluate_accuracy],
        experiment_name=f"First experiment ollama {timestamp}",
        experiment_description="Uses ollama backend",
        experiment_metadata={'model':"Ollama default", 
                             "prompt_template": None} ,
    #    repetitions=3
    #    concurrency=1
    #    dry_run=True,
        timeout=1000
    )
    
    print("Experiment complete. Check Phoenix UI in your browser.")


# Future to do items

Potentially changes to make evaluation more automated / efficient

1. Add method to Agent called something like get_metadata that can return a json of the current state of the software when running the benchmark.  Could include models used, prompt templates, git version?  Thoughts on what she be included here?
2. Make it easier to swap out critical components when working programmatically.  This could be how we switch between Ollama and samba nova.
3. Should reformatting be in the evaluation function? 
4. Speed up time needed to benchmark by exploring different backends.
5. Clean up what agent.run() returns to be only execution result once we have tracing in place? 
6. Further exploring UI (prompt playground; figure out if we can plug up sambanova)
7. 